### ✅ LLM을 통한 데이터 라벨링 보조(data_labeling(gpt 활용))

- 데이터 문제: 단답형으로 이루어진 대화턴이 많아 키워드 분석으로는 한계가 존재
- LLM을 통해 해당 발화가 어떤 상태인지 판단 필요
- 로컬 환경에서 GPU 필요없는 Gemma 3.gguf 사용(0.src 폴더에 저장)

#### 👉 LLM 로드(gemma3 활용)
- 사용 패키지 랭체인 활용
- llama 계열 -> chatgpt와 동일한 방식 적용 위해 ChatLlamaCpp 모듈 활용

In [ ]:
!pip install -r requirements.txt

In [1]:
# llama-cpp-python을 사용하는 GGUF 모델 로더
from langchain_community.chat_models import ChatLlamaCpp
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from typing import List,Annotated, Literal 

import pandas as pd

# ===  모델 생성 ====
model_path = "../model/gemma-3-1B-it-QAT-Q4_0.gguf"

gemma_llm = ChatLlamaCpp(
    model_path= model_path,
    n_ctx=4096,  # 컨텍스트 길이 설정 (길수록 더 많은 대화 기억)
    n_gpu_layers=35, # GPU 오프로드 레이어 수 (선택 사항, 성능 향상 목적)
    temperature=0.0,
    verbose=False,
)

print("\n===🎯Gemma3 테스트===")
print("질문: 안녕하세요")
print('답변: ', gemma_llm.invoke('안녕하세요?').content)

c:\Users\user\OneDrive\바탕 화면\새 폴더\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



===🎯Gemma3 테스트===
질문: 안녕하세요
답변:  안녕하세요! 무엇을 도와드릴까요? 😊



#### 👉 LLM에 프롬프트 적용하여 라벨링

In [2]:
# # 출력 형식 파서 정의
class judgement(BaseModel):
    answer: Annotated[List[Literal['positive', 'critical','danger' ,'emergency']], "AI 판단 결과로, 각 항목은 상태를 나타냅니다."]

parser = PydanticOutputParser(pydantic_object=judgement)

# 파서에서 형식 안내문 추출
format_instructions = parser.get_format_instructions()

# 1. 템플릿 문자열 정의: f-string 대신 일반 문자열을 사용하여 {format_instructions}를 '템플릿 변수'로 남깁니다.
system_prompt_str = """
당신은 홀로 거주하는 어르신과 AI의 대화 내용을 분석하는 전문가입니다.
당신의 임무는 대화 내용을 듣고, 어르신의 정서 상태나 위험 신호를 판단하여 공무원이 주의 깊게 살펴봐야 하는 정도를 다음 네 단계 중 하나로 분류하는 것입니다. 문장이 사람에게 해를 주는지를 분석하는 것이 아니라, 감정 카테고리만 판단합니다.

분류 단계:
분류 | 목표/조치 수준 | 핵심 판단 기준 (AI 분석 시 주력할 관찰 징후)
:---|:---|:---
positive | 안정 유지 / 일상적 관리 | 일상성 & 긍정적 정서: 명랑하고 활발한 어조, 대화에 적극적 참여, 일상이나 주변 환경에 대한 만족감 표현. (특이 사항 없음)
danger  | 주의 관찰 / 단기 모니터링 필요 | 정서적 불안정 초기: "심심하다," "적적하다," "잠이 잘 안 온다" 등 가벼운 외로움, 불안, 우울의 싹. 일회성 건강 염려 표현 (ex. "어깨가 좀 아프네"). 일시적이고 가벼운 문제 호소.
critical | 즉각 개입 / 전문가 상담 연계 필요 | 명확한 정서적 기능 저하: "살기 싫다," "사는 게 의미 없다" 등 지속적이고 반복적인 우울 및 체념, 무기력 표현. 사회적 단절을 명시적으로 언급. **식사/수면 패턴의 뚜렷한 장기적 변화**나 자신 돌봄 기능 저하 징후.
emergency | 긴급 구조 / 즉시 신체 안전 확인 | 생명 위험 및 인지 혼란: "죽고 싶다," "이제 끝내고 싶다" 등 자살에 대한 명확하고 구체적인 언급. 극도의 공포, 고립감 호소. **현실 인식 오류 (인지 저하)**나 심각한 신체적 응급 상황이 명백한 징후. (예: 호흡 곤란 호소, 심한 통증 호소)

분류 기준:
1. 어르신의 말투나 표현에 슬픔, 체념, 고립감, 신체적 이상, 인지 혼란 등이 있는지 확인하십시오.
2. 판단은 객관적으로 하며, 불필요한 추측이나 설명은 금지합니다.

**🚨 출력 규칙: 당신의 출력은 반드시 아래 JSON 스키마를 따르는 JSON 객체여야 합니다. JSON 객체 외의 어떠한 설명, 추론, 추가 텍스트도 포함해서는 안 됩니다. 'answer' 키의 값은 반드시 네 단계(positive, critical, danger, emergency) 중 하나를 포함하는 배열(List) 형태여야 합니다.**

{format_instructions}
"""

# 2. 템플릿 제작: system_prompt_str을 사용
template = ChatPromptTemplate.from_messages([
    ('system', system_prompt_str),  # {format_instructions} 템플릿 변수가 등록됨
    ('user', "{input}"),
])
llm_structured = gemma_llm.with_structured_output(judgement)

# 3. partial을 사용하여 format_instructions 변수에 값을 바인딩합니다.
# 이 코드가 실행되면 template의 input_variables에서 'format_instructions'가 제거됩니다.
template_final = template.partial(format_instructions=format_instructions)


# 4. 탬플릿과 연결(최종 LLM): template_final 사용
llm_chain = template_final | llm_structured 

# 5. 올바른 호출
print('===🎯예시 내용===')
q1 = '요새 내가 건강이 너무 안 좋아서 쓰러질것 같아 힘들어 죽겠어'
q2 = '이제 나는 여기까지 인가봐 살 기운이 없어'
print(f'질문: {q1}')
print(llm_chain.invoke({'input': q1}).answer[0])
print()
print(f'질문: {q2}')
print(llm_chain.invoke({'input': q2}).answer[0])

===🎯예시 내용===
질문: 요새 내가 건강이 너무 안 좋아서 쓰러질것 같아 힘들어 죽겠어
critical

질문: 이제 나는 여기까지 인가봐 살 기운이 없어
emergency


#### 👉 데이터 로드 후 라벨링 작업

In [3]:
PATH = r"../data/(원본데이터)발화데이터(대전중구).xlsx"

# 데이터 로드
df = pd.read_excel(PATH)
df = df.dropna().reset_index(drop=True)
print('기존데이터 구조')
print(df.shape)

# 파일 경로 불러오기 (10분단위 대화턴 묶음)
df_list = [] 
for y,sub in df.groupby(['doll_id']):

    sub['min'] = sub['uttered_at'].diff() > pd.Timedelta(minutes=10)
    # True가 나올 때마다 그룹이 바뀌도록 누적 그룹 번호 생성
    sub["group"] = sub["min"].cumsum()

    # False만 묶어서 텍스트 합치기
    merged = (
        sub.groupby(["group", "min"])
        .agg({
            "text": lambda x: " ".join(x) if not x.empty else "",
            "uttered_at": "first"
        })
        .reset_index(drop=True)
    )

    # 최종 결과
    merged['doll_id'] = y[0]
    df_list.append(merged)
merge_df = pd.concat(df_list)   
print('대화턴 합친 데이터 구조')
print(merge_df.shape)

기존데이터 구조
(32973, 3)
대화턴 합친 데이터 구조
(12471, 3)


In [ ]:
# 정답 리스트
answer_list =[]
for idx, x in enumerate(df['text']):
    try:
        answer = llm_chain.invoke(x)
    except:
        answer = 'positive' # 긍정으로 처리 -> 해당부분은 눈으로 한번더 확인해서 분류 예정
        print(f'{idx} 번째 오류')
    if idx == 0:
        print('결과 : ',answer.answer[0])
    answer_list.append(answer)  

final_list =[x.answer[0] for x in answer_list]
df['답변'] = final_list
df.to_excel("../data/(데이터라벨링)전체발화데이터(대전중구).xlsx", index=False, encoding="utf-8-sig")
print(f"✅ (데이터라벨링)전체발화데이터(대전중구).csv로 저장되었습니다.")

결과 :  critical
5348 번째 오류
6930 번째 오류


In [ ]:
# 정답 리스트
answer_list =[]
for idx, x in enumerate(merge_df['text']):
    try:
        answer = llm_chain.invoke(x)
    except:
        answer = 'positive' # 긍정으로 처리 -> 해당부분은 눈으로 한번더 확인해서 분류 예정
        print(f'{idx} 번째 오류')
    if idx == 0:
        print('결과 : ',answer.answer[0])
    answer_list.append(answer)  

final_list =[x.answer[0] for x in answer_list]
df['답변'] = final_list
df.to_excel("../data/(데이터라벨링)대화턴발화데이터(대전중구).xlsx", index=False, encoding="utf-8-sig")
print(f"✅ (데이터라벨링)대화턴발화데이터(대전중구).csv로 저장되었습니다.")